In [ ]:
"""Collect from comet-ml results using a list of experimental keys."""

# pylint: disable=import-error, redefined-outer-name, too-many-branches, unnecessary-lambda

## SETUP

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict

import pandas as pd
from comet_ml.api import API, APIExperiment
from tqdm import tqdm

from epiclass.utils.notebooks.paper.paper_utilities import (
    merge_renamed_columns,
    writable_path,
)

In [97]:
api = API()

## Initial download

In [ ]:
paper_dir = Path.home() / "Projects" / "epiclass" / "output" / "paper"
exp_list_path = paper_dir / "tables" / "training_experiment_keys.csv"
exp_df = pd.read_csv(exp_list_path, low_memory=False)
print(exp_df.shape)
print(exp_df.columns)

In [ ]:
exp_keys = set(exp_df["exp_key"].tolist())
print(len(exp_keys))

In [ ]:
# get_metadata(), direct
needed_meta = ["startTimeMillis", "projectName", "workspaceName"]

# get_git_metadata(), direct
needed_git_meta = ["parent"]

# get_others_summary(), elem
needed_others = [
    "category",
    "Code version / commit",
    "Name",
    "SLURM_JOB_ID",
    "Training time",
    "Total nb of files",
    "Training size",
    "Validation size",
    "train size",
    "validation size",
]

# get_parameters_summary(), elem
needed_parameters = [
    "input_size",
    "output_size",
    "run_arg_4",  # metadata filepath
    "run_arg_5",  # logdir filepath
]

# get_metrics_summary, elem
needed_metrics = [
    "Training time",
    "tra_Accuracy",
    "tra_F1Score",
    "val_Accuracy",
    "val_F1Score",
    "tra_MulticlassAccuracy",
    "tra_MulticlassF1Score",
    "val_MulticlassAccuracy",
    "val_MulticlassF1Score",
]

In [ ]:
def collect_exp_info(exp_key: str) -> Dict[str, Any]:
    """Collect experiment info from comet-ml using the API.

    Args:
        exp_key (str): Experiment key.

    Returns:
        Dict[str, Any]: Collected info.
    """
    exp_data: APIExperiment = api.get_experiment_by_key(exp_key)  # type: ignore
    collected_info: Dict[str, Any] = {}
    collected_info["Experiment key"] = exp_key

    # - Direct attributes -

    # Metadata
    metadata = exp_data.get_metadata()
    for attr in needed_meta:
        collected_info[attr] = metadata.get(attr, None)

    # Git metadata
    collected_info["Exact commit"] = exp_data.get_git_metadata()["parent"]

    # - Nested attributes -

    # Others summary
    others_summary = exp_data.get_others_summary()
    for elem in others_summary:
        name = elem["name"]
        if name in needed_others:
            collected_info[name] = elem.get("valueCurrent", None)

    # Parameters summary
    parameters_summary = exp_data.get_parameters_summary()
    for elem in parameters_summary:
        name = elem["name"]
        if name in needed_parameters:
            collected_info[name] = elem.get("valueCurrent", None)

    # Metrics summary
    metrics_summary = exp_data.get_metrics_summary()
    for elem in metrics_summary:
        name = elem["name"]
        if name in needed_metrics:
            collected_info[name] = elem.get("valueCurrent", None)

    return collected_info

In [ ]:
all_collected_info = {}
for exp_key in tqdm(exp_keys, desc="Experiments", unit="exp"):
    all_collected_info[exp_key] = collect_exp_info(exp_key)

## Formatting

In [ ]:
collected_df = pd.DataFrame.from_dict(all_collected_info, orient="index")
print(collected_df.columns)

In [ ]:
# Logged names changed over time. Depending on the runs, either name can be absent.
renamed_cols = {
    "Training size": "train size",
    "tra_MulticlassAccuracy": "tra_Accuracy",
    "tra_MulticlassF1Score": "tra_F1Score",
    "val_MulticlassAccuracy": "val_Accuracy",
    "val_MulticlassF1Score": "val_F1Score",
}
collected_df = merge_renamed_columns(collected_df, renamed_cols)

In [ ]:
# Get date

# Server start time is in milliseconds, convert to seconds
unix_time = round(collected_df["startTimeMillis"] / 1000)

# Convert to date
datetime = pd.to_datetime(unix_time, unit="s")
collected_df["Date (YYYY-MM-DD)"] = datetime.dt.date

collected_df.rename(
    columns={"startTimeMillis": "Server start time (Unix, ms)"}, inplace=True
)

In [ ]:
# Change for a new list of experiments, existing files are not overwritten.
output_name = "collected_experiments_info_cometml"

column_order = [
    "Name",
    "category",
    "output_size",
    "Total nb of files",
    "train size",
    "validation size",
    "tra_Accuracy",
    "tra_F1Score",
    "val_Accuracy",
    "val_F1Score",
    "Experiment key",
    "SLURM_JOB_ID",
    "Code version / commit",
    "Exact commit",
    "Server start time (Unix, ms)",
    "Date (YYYY-MM-DD)",
]

collected_df[column_order].to_csv(
    writable_path(paper_dir / "tables" / f"{output_name}.csv"), index=False
)

## Supplementary table format

Same format as the "MLP training details" supplementary table.

In [ ]:
def training_time_to_seconds(value: Any) -> float:
    """Convert a logged training time to seconds.

    Handles seconds (949.0), H:MM:SS and datetime.timedelta(days=X, seconds=Y) values.
    """
    if pd.isna(value):
        return float("nan")
    value = str(value)
    timedelta_search = re.search(r"timedelta\((?:days=(\d+), )?seconds=(\d+)", value)
    if timedelta_search:
        days, seconds = timedelta_search.groups()
        return int(days or 0) * 86400 + int(seconds)
    if ":" in value:
        return pd.to_timedelta(value).total_seconds()
    return float(value)


def format_training_details(collected_df: pd.DataFrame) -> pd.DataFrame:
    """Format collected info like the "MLP training details" supplementary table.

    Args:
        collected_df (pd.DataFrame): One row per experiment, with merged renamed columns
            and date columns.

    Returns:
        pd.DataFrame: The supplementary table columns, one row per experiment.
    """
    df = collected_df.copy()

    # Run description
    df["Dataset"] = (
        df["run_arg_4"]
        .str.contains("dfreeze-v2/", regex=False)
        .map({True: "dfreeze_v2", False: "2023-01-epiatlas-freeze"})
    )
    # The last match is the feature set, e.g. .../hg38_epiatlas-freeze-v2/hg38_100kb_all_none/...
    df["feature_set_name"] = df["run_arg_5"].str.findall(r"/(hg38_[^/]+)").str[-1]
    df["metadata_category"] = df["category"]
    df["split"] = df["Name"].str.extract(r"(split\d+|complete_no_valid(?:_oversample)?)")[
        0
    ]

    # Numeric values
    for col in ["input_size", "output_size", "Total nb of files", "train size"]:
        df[col] = pd.to_numeric(df[col]).astype("Int64")

    metric_cols = ["tra_Accuracy", "tra_F1Score", "val_Accuracy", "val_F1Score"]
    df[metric_cols] = df[metric_cols].apply(pd.to_numeric).round(4).astype(object)

    training_seconds = df["Training time"].map(training_time_to_seconds)
    df["training time (minutes)"] = (training_seconds / 60).round().astype("Int64")

    # Validation size is not logged by the oldest runs, and is 0 without a validation set.
    validation_size = pd.to_numeric(df["validation size"]).astype("Int64")
    no_validation = validation_size == 0
    df["validation size"] = validation_size.astype(object).where(
        validation_size.notna(), "Unknown"
    )
    df.loc[no_validation, ["validation size", "val_Accuracy", "val_F1Score"]] = "NA"

    df["comet-url"] = (
        "https://www.comet.com/"
        + df["workspaceName"]
        + "/"
        + df["projectName"]
        + "/"
        + df["Experiment key"]
    )

    column_order = [
        "Dataset",
        "feature_set_name",
        "metadata_category",
        "split",
        "Name",
        "input_size",
        "output_size",
        "Total nb of files",
        "train size",
        "validation size",
        "tra_Accuracy",
        "tra_F1Score",
        "val_Accuracy",
        "val_F1Score",
        "training time (minutes)",
        "Experiment key",
        "Code version / commit",
        "Exact commit",
        "Server start time (Unix, ms)",
        "Date (YYYY-MM-DD)",
        "comet-url",
    ]
    sort_cols = ["Dataset", "feature_set_name", "metadata_category", "split"]
    return df[column_order].sort_values(sort_cols).reset_index(drop=True)

In [ ]:
training_details_df = format_training_details(collected_df)
training_details_df.to_csv(
    writable_path(paper_dir / "tables" / f"{output_name}_training_details.csv"),
    index=False,
)